In [10]:
import jupyter_black
import torch
import os
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision import datasets, transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader

jupyter_black.load()

In [16]:
devices = "mps" if torch.backends.mps.is_available() else "CPU"
devices

'mps'

## Getting the data for health/disease detection

In [14]:
#import kagglehub

#path = kagglehub.dataset_download("andresmgs/plantdec")
#print("Path to dataset files:", path)

100%|██████████████████████████████████████| 74.1M/74.1M [00:04<00:00, 16.4MB/s]

Extracting files...


Path to dataset files: /Users/avnay/Desktop/AIML/Plant_project/leafsnap_data/datasets/andresmgs/plantdec/versions/7


In [5]:
#import shutil

#shutil.copytree(path, "Plant_project", dirs_exist_ok=True)
#print("Data successfully copied to your working folder!")

Data successfully copied to your working folder!


### fixing the data 

In [5]:
class CustomPlantDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (Path/str): Path to the split folder (e.g., 'Plant_project_data/train')
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.root_dir = Path(root_dir)
        self.images_dir = self.root_dir / "images"
        self.labels_dir = self.root_dir / "labels"

        # List all image files sorted so they align correctly
        self.image_files = sorted(
            [
                f
                for f in os.listdir(self.images_dir)
                if f.endswith((".jpg", ".jpeg", ".png"))
            ]
        )
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # 1. Get image path and load it
        img_name = self.image_files[idx]
        img_path = self.images_dir / img_name
        image = Image.open(img_path).convert("RGB")

        # 2. Get corresponding label file path
        label_name = img_name.rsplit(".", 1)[0] + ".txt"
        label_path = self.labels_dir / label_name

        # 3. Read the class ID from the label file (YOLO format: class_id x_center y_center width height)
        # We take the first integer on the first line as the classification label
        label = 0  # Default fallback
        if label_path.exists() and os.path.getsize(label_path) > 0:
            with open(label_path, "r") as f:
                first_line = f.readline().strip().split()
                if first_line:
                    label = int(first_line[0])  # Grab the class ID

        # 4. Apply any transformations
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [7]:
base_data_path = Path("Plant_project_data")

# 2. Append the subfolders
train_dir = base_data_path / "train"
valid_dir = base_data_path / "valid"
test_dir = base_data_path / "test"
data_transform = transforms.Compose(
    [transforms.Resize(size=(64, 64)), transforms.ToTensor()]
)
doc_train_data = CustomPlantDataset(
    root_dir="Plant_project_data/train", transform=data_transform
)
doc_valid_data = CustomPlantDataset(
    root_dir="Plant_project_data/valid", transform=data_transform
)
doc_test_data = CustomPlantDataset(
    root_dir="Plant_project_data/test", transform=data_transform
)

In [11]:
BATCH_SIZE = 32
train_dataloader = DataLoader(
    dataset=doc_train_data, batch_size=BATCH_SIZE, shuffle=True
)
valid_dataloader = DataLoader(
    dataset=doc_valid_data, batch_size=BATCH_SIZE, shuffle=False
)
test_dataloader = DataLoader(
    dataset=doc_test_data, batch_size=BATCH_SIZE, shuffle=False
)

## Getting the data for species identification

In [15]:
#os.environ["KAGGLEHUB_CACHE"] = str(Path("").absolute() / "leafsnap_data")

# 2. Run the download (it will go straight into the folder above)
#path = kagglehub.dataset_download("vandat2601/leafsnap-processed")
#print("Downloaded straight to:", path)

# 3. See what folders are inside it
#print("\nContents of your local folder:")
#print(os.listdir(path))

100%|████████████████████████████████████████| 137M/137M [00:09<00:00, 15.8MB/s]

Extracting files...


Downloaded straight to: /Users/avnay/Desktop/AIML/Plant_project/leafsnap_data/datasets/vandat2601/leafsnap-processed/versions/1

Contents of your local folder:
['test', 'train', 'val']
